In [1]:
#Import all packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="unknown class")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Load the .npy embeddings + metadata, and confirm row alignment
X_span = np.load("bert_span_embeddings.npy")
X_cls  = np.load("bert_cls_embeddings.npy")
meta   = pd.read_csv("bert_metadata.csv")

print("X_span shape:", X_span.shape)
print("X_cls  shape:", X_cls.shape)
print("meta  rows  :", len(meta))
print("First TokenIDs:", meta["TokenID"].head().tolist())

X_span shape: (594, 768)
X_cls  shape: (594, 768)
meta  rows  : 594
First TokenIDs: ['40001018', '40001024', '40003002', '40003017', '40004012']


In [3]:
# Helper to extract component grams from composite ones

def extract_components(gram_string, paradigms_for_lang):
    """
    Takes a gram string (e.g. 'past participle') and a list of paradigms
    (e.g. [['present','past'], ['participle']]) and returns a list of
    components that occur in the gram.
    """
    gram_string = str(gram_string).lower()
    components = []
    for paradigm in paradigms_for_lang:
        for p in paradigm:
            if p in gram_string:
                components.append(p)
    return sorted(set(components)) if components else ['NONE']

In [4]:
# === Load and prepare data ===

# Load the formatted CSV
df = pd.read_csv(
    "p1_run_ma.csv",
    dtype=str,
    keep_default_na=False,
    na_values=[]
)

# Define feature and gram columns
# NOTE: FEATURE_COLS are used in the feature-based pipeline, but NOT in the BERT-based pipeline
# FEATURE_COLS = ["TokenID", "DtExp", "DtRes", "DtUniv", "DtDTAM", "TLTS", "TLAspPfv", "TLMdIrr", "ClNegPol",
#                 "ClIntgv", "ClCntfcl", "VcPass", "ASTrns", "ASUnrg", "ASUncc", "AktState", "AktAct", "AktAcc", "AktAch"]

GRAM_COLS = ["eng_gram", "deu_gram", "cmn_gram"]

# Quick shape check
print("Data loaded:", df.shape)
print("Columns:", df.columns.tolist()[:10], "...")

Data loaded: (594, 65)
Columns: ['TokenID', 't_o', 'FullText', 'Target', 't_surf', 'DtExp', 'CS', 'Reason', 'DtRes', 'CS.1'] ...


In [5]:
# === Define paradigms and extract components ===

paradigms = {
    'eng': [['participle'], ['present', 'past', 'infinitive', 'modal'], ['can', 'could', 'may', 'might', 'must', 'ought_to', 'shall', 'should', 'will', 'would']],
    'deu': [['partizip'], ['inf', 'prasens', 'prat'], ['indikativ', 'konj1', 'konj2'], ['duerfen', 'koennen', 'moegen', 'muessen', 'sollen', 'wollen'], ['werden'], ['haben', 'sein']],
    'cmn': [['le', 'guo'], ['adverbial'], ['yi', 'yijing', 'zaojiu', 'zaoyi'], ['suo_de'], ['compound'], ['you'], ['nullmarked']],
    'jpn': [['ta', 'te', 'simple'], ['iru'], ['shimau'], ['koto_ga_aru'], ['bakari', 'tokoro'], ['aru']]
}

# === Higher-level paradigm groupings ===
paradigm_groups = {
    'eng': {
        'modal': ['can', 'could', 'may', 'might', 'must', 'ought_to', 'shall', 'should', 'will', 'would']
    },
    'deu': {
        'modal': ['duerfen', 'koennen', 'moegen', 'muessen', 'sollen', 'wollen']
    },
    'cmn': {
        'adverbial': ['yi', 'yijing', 'zaojiu', 'zaoyi']
    }
}

# Make copies to avoid overwriting raw data
df_proc = df.copy()

# Apply component extraction to each gram column
for lg in ['eng', 'deu', 'cmn']:
    col = f"{lg}_gram"
    if col in df_proc.columns:
        df_proc[col] = df_proc[col].apply(lambda x: extract_components(x, paradigms[lg]))

# Quick peek
print(df_proc[[c for c in df_proc.columns if 'gram' in c]].head())

                eng_gram    cmn_gram  \
0     [participle, past]        [le]   
1     [participle, past]    [suo_de]   
2     [participle, past]        [le]   
3  [participle, present]    [suo_de]   
4     [participle, past]  [compound]   

                                    deu_gram jpn_gram  
0          [indikativ, partizip, prat, sein]      n_a  
1         [haben, indikativ, partizip, prat]       ta  
2       [indikativ, partizip, prasens, sein]       ta  
3      [haben, indikativ, partizip, prasens]      n_a  
4  [indikativ, partizip, prat, sein, werden]       ta  


In [6]:
# === Evaluation mode per paradigm ===
# keys are tuples that exactly match a paradigm list (order-insensitive; we'll normalize)
EVAL_MODES = {
    'eng': {
        ('participle',): 'binary',
        ('present','past','infinitive','modal'): 'multiclass_required',
        ('can','could','may','might','must','ought_to','shall','should','will','would'): 'multiclass_optional',
    },
    'deu': {
        ('partizip',): 'binary',
        ('inf','prasens','prat'): 'multiclass_required',
        ('indikativ', 'kon1', 'kon2'): 'multiclass_required',
        ('modal'): 'binary',
        ('duerfen', 'koennen', 'moegen', 'muessen', 'sollen', 'wollen'): 'multiclass_optional',
        ('werden'): 'binary',
        ('haben', 'sein'): 'multiclass_optional',
    },
    'cmn': {
        ('le','guo'): 'multiclass_optional',
        ('adverbial'): 'binary',
        ('yi','yijing','zaojiu','zaoyi'): 'multiclass_optional',
        ('suo_de'): 'binary',
        ('compound'): 'binary',
        ('you'): 'binary',
        ('nullmarked'): 'binary',
        # Note: 'nullmarked' is a label in the data; we usually *don’t* use it as a paradigm here
    },
    'jpn': {
        ('ta', 'te', 'simple'): 'multiclass_required',
        ('iru'): 'binary',
        ('shimau'): 'binary',
        ('koto_ga_aru'): 'binary',
        ('bakari', 'tokoro'): 'multiclass_optional',
        ('aru'): 'binary'    
}}

# helper to normalize a paradigm list into a tuple key
def norm_key(paradigm_list):
    return tuple(sorted(paradigm_list))

In [7]:
# === Helper: generate y_train / y_dev targets according to evaluation mode ===
def build_targets(lang_full, paradigm, mode):
    """
    Returns numeric or categorical targets for this paradigm.

    mode = 'binary'              → 0/1 labels
    mode = 'multiclass_required' → one label from paradigm (string)
    mode = 'multiclass_optional' → one label or '0' for none
    """
    if mode == 'binary':
        # 1 if any element of paradigm occurs, else 0
        y_train = y_train_no_none[lang_full].apply(lambda lst: int(any(p in lst for p in paradigm)))
        y_dev   = y_dev_no_none[lang_full].apply(lambda lst: int(any(p in lst for p in paradigm)))
        return y_train, y_dev

    elif mode == 'multiclass_required':
        # exactly one of the paradigm members should appear
        def get_label(lst):
            for p in paradigm:
                if p in lst:
                    return p
            # if none appear (should be rare)
            return 'MISSING'
        y_train = y_train_no_none[lang_full].apply(get_label)
        y_dev   = y_dev_no_none[lang_full].apply(get_label)
        return y_train, y_dev

    elif mode == 'multiclass_optional':
        # one label if present, otherwise '0' for absence
        def get_label(lst):
            for p in paradigm:
                if p in lst:
                    return p
            return '0'
        y_train = y_train_no_none[lang_full].apply(get_label)
        y_dev   = y_dev_no_none[lang_full].apply(get_label)
        return y_train, y_dev

    else:
        raise ValueError(f"Unknown mode: {mode}")


In [8]:
# === Split into train/dev sets (stratified, reproducible) ===

# X is now BERT embeddings + TokenID
X = meta.copy()  # meta comes from bert_metadata.csv (Cell 2)
X_emb = pd.DataFrame(X_span)     # use X_cls here if you want the CLS baseline
X = pd.concat([X, X_emb], axis=1)
y = df_proc[["eng_gram", "deu_gram", "cmn_gram"]]

# For stratification, we need a single column of labels per row — use German for now
# (can change this to another language if preferred)
stratify_col = y["deu_gram"].apply(lambda x: "_".join(x))

# --- Handle rare labels (classes with only one example cannot be stratified) ---
vc = stratify_col.value_counts()
rare_labels = vc[vc < 2].index

if len(rare_labels) > 0:
    print(f"⚠️ Removing {len(rare_labels)} rare classes (1 occurrence each).")
    mask = ~stratify_col.isin(rare_labels)
    X = X.loc[mask].reset_index(drop=True)
    y = y.loc[mask].reset_index(drop=True)
    stratify_col = stratify_col.loc[mask].reset_index(drop=True)
    print(f"Remaining samples after cleanup: {len(stratify_col)}")
else:
    print("✅ No rare classes detected — proceeding with full dataset.")

# --- Perform the stratified, reproducible split ---
X_train, X_dev, y_train, y_dev = train_test_split(
    X, y,
    test_size=0.5,
    stratify=stratify_col,
    random_state=42   # ensures reproducible splits
)

print("\nStratified train/dev split complete:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)

⚠️ Removing 12 rare classes (1 occurrence each).
Remaining samples after cleanup: 582

Stratified train/dev split complete:
  X_train: (291, 769)
  X_dev: (291, 769)


In [9]:
# === Filter out rows with 'NONE' or 'n_a' labels (DEU and CMN only) ===

X_train_no_none = {}
X_dev_no_none   = {}
y_train_no_none = {}
y_dev_no_none   = {}

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    # Define filtering rule per language
    if lang in ["deu_gram", "cmn_gram"]:
        invalid = lambda lbls: any(lbl in ['NONE', 'n_a'] for lbl in lbls)
    else:
        invalid = lambda lbls: lbls == ['NONE']

    mask_train = ~y_train[lang].apply(invalid)
    mask_dev   = ~y_dev[lang].apply(invalid)
    
    X_train_no_none[lang] = X_train.loc[mask_train].reset_index(drop=True)
    X_dev_no_none[lang]   = X_dev.loc[mask_dev].reset_index(drop=True)
    y_train_no_none[lang] = y_train.loc[mask_train, lang].reset_index(drop=True)
    y_dev_no_none[lang]   = y_dev.loc[mask_dev, lang].reset_index(drop=True)

for lang in y_train_no_none:
    print(f"{lang}: {len(y_train_no_none[lang])} train rows, {len(y_dev_no_none[lang])} dev rows")

eng_gram: 290 train rows, 291 dev rows
deu_gram: 257 train rows, 258 dev rows
cmn_gram: 281 train rows, 280 dev rows


In [10]:
# === Binarize label sets per language'  ===

"""
General Baseline
These train and evaluate language-level models on the full gram set.
They don’t interfere with the new paradigm logic.
They can be left in as reference / baseline results, or commented out to save runtime.
"""


mlbs = {}
y_train_bin = {}
y_dev_bin   = {}

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    mlb = MultiLabelBinarizer()
    y_train_bin[lang] = mlb.fit_transform(y_train_no_none[lang])
    y_dev_bin[lang]   = mlb.transform(y_dev_no_none[lang])
    mlbs[lang] = mlb

    print(f"{lang}: {len(mlb.classes_)} labels → {mlb.classes_}")

eng_gram: 7 labels → ['could' 'might' 'participle' 'past' 'present' 'should' 'would']
deu_gram: 11 labels → ['haben' 'indikativ' 'koennen' 'konj1' 'konj2' 'partizip' 'prasens' 'prat'
 'sein' 'sollen' 'werden']
cmn_gram: 10 labels → ['compound' 'guo' 'le' 'nullmarked' 'suo_de' 'yi' 'yijing' 'you' 'zaojiu'
 'zaoyi']


In [11]:
# === Build paradigm-specific subsets (testing stage) ===

for lang_short, lang_full in zip(['eng', 'deu', 'cmn'], ['eng_gram', 'deu_gram', 'cmn_gram']):
    print(f"\n=== Paradigm subsets for {lang_short.upper()} ===")
    for paradigm in paradigms[lang_short]:
        pname = "_".join(paradigm)  # just for readable naming
        mask = y_dev_no_none[lang_full].apply(lambda lst: any(p in lst for p in paradigm))
        subset_size = mask.sum()
        print(f"{pname:<20} → {subset_size} tokens in dev set")


=== Paradigm subsets for ENG ===
participle           → 288 tokens in dev set
present_past_infinitive_modal → 277 tokens in dev set
can_could_may_might_must_ought_to_shall_should_will_would → 13 tokens in dev set

=== Paradigm subsets for DEU ===
partizip             → 238 tokens in dev set
inf_prasens_prat     → 258 tokens in dev set
indikativ_konj1_konj2 → 258 tokens in dev set
duerfen_koennen_moegen_muessen_sollen_wollen → 2 tokens in dev set
werden               → 16 tokens in dev set
haben_sein           → 243 tokens in dev set

=== Paradigm subsets for CMN ===
le_guo               → 132 tokens in dev set
adverbial            → 0 tokens in dev set
yi_yijing_zaojiu_zaoyi → 34 tokens in dev set
suo_de               → 24 tokens in dev set
compound             → 30 tokens in dev set
you                  → 16 tokens in dev set
nullmarked           → 89 tokens in dev set


In [12]:
# === Train and evaluate within each paradigm subset (MLP Fixed Logic) ===

all_preds_mlp = []

for lang_short, lang_full in zip(['eng', 'deu', 'cmn'], ['eng_gram', 'deu_gram', 'cmn_gram']):
    print(f"\n=== Paradigm-level classification for {lang_short.upper()} ===")
    for paradigm in paradigms[lang_short]:
        paradigm_key = tuple(paradigm)
        pname = "_".join(paradigm_key)

        mode = EVAL_MODES[lang_short].get(
            paradigm_key,
            EVAL_MODES[lang_short].get(pname, 'multiclass_optional')
        )

        # === Determine Subsetting based on Mode ===
        if mode in ['binary', 'multiclass_optional']:
            X_train_sub = X_train_no_none[lang_full].drop(columns=["TokenID"], errors='ignore')
            X_dev_sub   = X_dev_no_none[lang_full].drop(columns=["TokenID"], errors='ignore')
            tokenid_dev = X_dev_no_none[lang_full]["TokenID"].reset_index(drop=True)

            y_train_sub, y_dev_sub = build_targets(lang_full, paradigm, mode)

        elif mode == 'multiclass_required':
            mask_train = y_train_no_none[lang_full].apply(lambda lst: any(p in lst for p in paradigm))
            mask_dev   = y_dev_no_none[lang_full].apply(lambda lst: any(p in lst for p in paradigm))

            X_train_sub = X_train_no_none[lang_full].loc[mask_train].drop(columns=["TokenID"], errors='ignore')
            X_dev_sub   = X_dev_no_none[lang_full].loc[mask_dev].drop(columns=["TokenID"], errors='ignore')
            tokenid_dev = X_dev_no_none[lang_full].loc[mask_dev, "TokenID"].reset_index(drop=True)

            y_train_all, y_dev_all = build_targets(lang_full, paradigm, mode)
            y_train_sub = y_train_all.loc[mask_train].reset_index(drop=True)
            y_dev_sub   = y_dev_all.loc[mask_dev].reset_index(drop=True)

        # Skip tiny sets
        if len(y_train_sub) < 5 or len(y_dev_sub) < 5:
            continue

        # === SCALE FEATURES (MLP NEEDS THIS) ===
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_sub)
        X_dev_scaled   = scaler.transform(X_dev_sub)

        # === Train MLP ===
        clf = MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            max_iter=500,
            random_state=42
        )
        clf.fit(X_train_scaled, y_train_sub)
        preds = clf.predict(X_dev_scaled)

        # --- Collect predictions for error analysis (MLP) ---
        pred_df = pd.DataFrame({
            "TokenID": tokenid_dev,
            "language": lang_short,
            "paradigm": pname,
            "mode": mode,
            "gold": y_dev_sub,
            "pred": preds
        })
        pred_df["correct"] = pred_df["gold"] == pred_df["pred"]
        all_preds_mlp.append(pred_df)

        # === Evaluate ===
        print(f"\n  {pname:<35} → {len(y_dev_sub):>3} dev tokens | mode = {mode}")

        if mode == 'binary':
            acc = (preds == y_dev_sub).mean()
            print(f"    Binary accuracy = {acc:.3f}")
        else:
            print(classification_report(y_dev_sub, preds, zero_division=0))


=== Paradigm-level classification for ENG ===

  participle                          → 291 dev tokens | mode = binary
    Binary accuracy = 0.990

  present_past_infinitive_modal       → 277 dev tokens | mode = multiclass_required
              precision    recall  f1-score   support

        past       0.99      0.97      0.98        72
     present       0.99      1.00      0.99       205

    accuracy                           0.99       277
   macro avg       0.99      0.98      0.99       277
weighted avg       0.99      0.99      0.99       277


  can_could_may_might_must_ought_to_shall_should_will_would → 291 dev tokens | mode = multiclass_optional
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       278
       could       1.00      1.00      1.00         1
      should       0.50      0.33      0.40         3
       would       0.90      1.00      0.95         9

    accuracy                           0.99       291
   macro

In [13]:
# === EXPORT: MLP error analysis artifacts ===
# NOTE: Exports paradigm-level MLP predictions for BERT-based experiments
import os

ERROR_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis"
os.makedirs(ERROR_DIR, exist_ok=True)

mlp_preds_long = pd.concat(all_preds_mlp, ignore_index=True)
mlp_preds_long.to_csv(f"{ERROR_DIR}/y_pred_mlp_bert.csv", index=False)
X_dev.to_csv(f"{ERROR_DIR}/X_dev_bert.csv", index=False)

In [14]:
# === Sanity Check ===
df = pd.read_csv("/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/y_pred_mlp_long_hm.csv")

print(df.head())
print(df["language"].value_counts())
print(df["mode"].value_counts())
print("Accuracy:", df["correct"].mean())

     TokenID language    paradigm    mode gold pred  correct
0   43008029      eng  participle  binary    1    1     True
1   43011017      eng  participle  binary    1    1     True
2   42023015      eng  participle  binary    1    1     True
3  42001068b      eng  participle  binary    1    1     True
4   43009022      eng  participle  binary    1    1     True
language
cmn    1120
deu     912
eng     500
Name: count, dtype: int64
mode
binary                 1274
multiclass_optional     946
multiclass_required     312
Name: count, dtype: int64
Accuracy: 0.8906003159557662
